# 02 — Data Preparation
## CRISP-DM Phase: Data Preparation

---

| | |
|---|---|
| **Phase** | Data Preparation |
| **Goal** | Join all 9 Olist tables into a single customer-level master table; construct the binary churn label; export a clean, validated Parquet artifact for EDA and feature engineering. |
| **Inputs** | Raw CSV files downloaded via `kagglehub` (same path as NB01); confirmed dtype decisions and data quality findings from `01_data_understanding.ipynb` |
| **Outputs** | `outputs/02_master_table.parquet` — one row per eligible `customer_unique_id`, all aggregated features, churn label (0/1) |

---

### Notebook Road-map

This notebook executes a strict 10-step pipeline. Each step is documented with a dedicated markdown cell explaining *what* is being done and *why*. After every join, the shape is printed before and after so that any unexpected row count change is immediately visible.

| Step | Action | Output variable |
|---|---|---|
| 1 | Aggregate payments → order level | `payments_agg` |
| 2 | Deduplicate geolocation → zip→state map | `geo_map` |
| 3 | Deduplicate reviews → order level | `reviews_agg` |
| 4 | Aggregate order items → order level | `items_agg` |
| 5 | Join products + category translation | `items_agg` (enriched) |
| 6 | Join sellers + geo map | `sellers_geo` |
| 7 | Build master table (delivered orders, left-join outward) | `master` |
| 8 | Resolve to `customer_unique_id` level (last order per customer) | `master` |
| 9 | Construct churn label (vectorised) | `master['churn']` |
| 10 | Validate all checks; export Parquet | `outputs/02_master_table.parquet` |

---

### ⚠️ Critical Design Principle: `customer_id` vs `customer_unique_id`

This is the single most important data integrity rule in this notebook.

- **`customer_id`** is *order-scoped*: each order row in the `orders` table carries a distinct `customer_id` even if the same physical customer placed multiple orders. It is a surrogate key for the order-customer pair, not for the customer.
- **`customer_unique_id`** is the true customer identifier that persists across orders.

**Consequence:** every `groupby`, aggregation, and final master table index must use `customer_unique_id`. The `customer_id` field is used *only* as the join key between `customers` and `orders` — it is dropped from the final table.

Failing to make this distinction would cause a customer with 3 orders to appear as 3 distinct "customers", inflating the population size and making churn labels impossible to compute correctly.


## 0. Imports and Environment Setup

In [1]:
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Ensure outputs directory exists
os.makedirs('outputs', exist_ok=True)

print("Environment ready.")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")


Environment ready.
pandas  : 2.3.3
numpy   : 2.3.5


**Interpretation:** Standard library imports. All intermediate variables are named consistently with the 10-step plan above. `outputs/` directory is created if it does not exist — this is where `02_master_table.parquet` will be written at the end of the notebook.

## 1. Load Raw CSV Files

All 9 tables are loaded with the confirmed dtype decisions from NB01:
- ID columns → `str` (prevents integer coercion of leading-zero zip codes and hash-like IDs)
- Zip code columns → `str` (preserves leading zeros — e.g. zip `01310` must not become `1310`)
- Timestamp columns → `datetime64[ns]` via `parse_dates`

The `kagglehub` download path is re-used from NB01. If the data is already cached locally, `kagglehub` serves it from cache without re-downloading.


In [2]:
import kagglehub

DATA_PATH = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(f"Data path: {DATA_PATH}")
print("Files available:")
for f in sorted(os.listdir(DATA_PATH)):
    print(f"  {f}")


Data path: /Users/seihavat/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2
Files available:
  olist_customers_dataset.csv
  olist_geolocation_dataset.csv
  olist_order_items_dataset.csv
  olist_order_payments_dataset.csv
  olist_order_reviews_dataset.csv
  olist_orders_dataset.csv
  olist_products_dataset.csv
  olist_sellers_dataset.csv
  product_category_name_translation.csv


**Interpretation:** `kagglehub` returns the local cache path. All 9 CSV files should be listed. If any file is missing, the notebook will raise a clear `FileNotFoundError` in the next cell rather than silently producing wrong results.

In [3]:
# ── Helper: build full path ──────────────────────────────────────────────
def fp(filename):
    return os.path.join(DATA_PATH, filename)

# ── Load orders ───────────────────────────────────────────────────────────
orders = pd.read_csv(
    fp('olist_orders_dataset.csv'),
    dtype={'order_id': str, 'customer_id': str},
    parse_dates=[
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ]
)
print(f"orders          : {orders.shape}")

# ── Load order_items ──────────────────────────────────────────────────────
order_items = pd.read_csv(
    fp('olist_order_items_dataset.csv'),
    dtype={'order_id': str, 'product_id': str, 'seller_id': str}
)
print(f"order_items     : {order_items.shape}")

# ── Load order_payments ───────────────────────────────────────────────────
order_payments = pd.read_csv(
    fp('olist_order_payments_dataset.csv'),
    dtype={'order_id': str, 'payment_type': str}
)
print(f"order_payments  : {order_payments.shape}")

# ── Load order_reviews ────────────────────────────────────────────────────
order_reviews = pd.read_csv(
    fp('olist_order_reviews_dataset.csv'),
    dtype={'review_id': str, 'order_id': str},
    parse_dates=['review_creation_date', 'review_answer_timestamp']
)
print(f"order_reviews   : {order_reviews.shape}")

# ── Load customers ────────────────────────────────────────────────────────
customers = pd.read_csv(
    fp('olist_customers_dataset.csv'),
    dtype={
        'customer_id': str,
        'customer_unique_id': str,
        'customer_zip_code_prefix': str,
    }
)
print(f"customers       : {customers.shape}")

# ── Load sellers ──────────────────────────────────────────────────────────
sellers = pd.read_csv(
    fp('olist_sellers_dataset.csv'),
    dtype={'seller_id': str, 'seller_zip_code_prefix': str}
)
print(f"sellers         : {sellers.shape}")

# ── Load products ─────────────────────────────────────────────────────────
products = pd.read_csv(
    fp('olist_products_dataset.csv'),
    dtype={'product_id': str, 'product_category_name': str}
)
print(f"products        : {products.shape}")

# ── Load geolocation ──────────────────────────────────────────────────────
geolocation = pd.read_csv(
    fp('olist_geolocation_dataset.csv'),
    dtype={'geolocation_zip_code_prefix': str, 'geolocation_state': str}
)
print(f"geolocation     : {geolocation.shape}")

# ── Load category translation ─────────────────────────────────────────────
cat_translation = pd.read_csv(fp('product_category_name_translation.csv'))
print(f"cat_translation : {cat_translation.shape}")


orders          : (99441, 8)
order_items     : (112650, 7)
order_payments  : (103886, 5)
order_reviews   : (99224, 7)
customers       : (99441, 5)
sellers         : (3095, 4)
products        : (32951, 9)
geolocation     : (1000163, 5)
cat_translation : (71, 2)


**Interpretation:** All 9 tables are loaded. The confirmed shapes from NB01 are reproduced here as a sanity check:

| Table | Expected rows |
|---|---|
| `orders` | 99,441 |
| `order_items` | 112,650 |
| `order_payments` | ~104K |
| `order_reviews` | ~100K |
| `customers` | ~100K |
| `sellers` | 3,095 |
| `products` | 32,951 |
| `geolocation` | ~1M |
| `cat_translation` | ~71 |

Any significant deviation from these counts warrants investigation before proceeding.


---
## Step 1 — Aggregate Payments to Order Level

### What and Why

The `order_payments` table has **multiple rows per order**. A single order can be split across payment methods — for example, a customer pays partly with a voucher and the remainder with a credit card. If we join the raw payments table directly to orders, each order would be duplicated once per payment row, inflating the master table.

**Mitigation:** aggregate payments to one row per `order_id` *before* any join.

**Aggregation specification (from NB01 confirmed decisions):**

| New column | Source column | Aggregation |
|---|---|---|
| `total_payment_value` | `payment_value` | `sum` |
| `max_installments` | `payment_installments` | `max` |
| `n_payment_types` | `payment_type` | `nunique` |
| `primary_payment_type` | `payment_type` | mode (most common type for the order) |
| `used_voucher` | `payment_type` | 1 if any row == `'voucher'`, else 0 |

**Note on zero-installment rows:** `payment_installments == 0` occurs for voucher payments. These are valid entries — do not impute. The `used_voucher` flag already captures this case explicitly.


In [5]:
print(f"order_payments BEFORE aggregation : {order_payments.shape}")

# ── used_voucher flag (before groupby so it sees all rows per order) ───────
order_payments['_is_voucher'] = (order_payments['payment_type'] == 'voucher').astype(int)

# ── mode helper (returns first mode value as scalar) ─────────────────────
def mode_first(s):
    m = s.mode()
    return m.iloc[0] if len(m) > 0 else np.nan

payments_agg = (
    order_payments
    .groupby('order_id', as_index=False)
    .agg(
        total_payment_value  = ('payment_value',        'sum'),
        max_installments     = ('payment_installments', 'max'),
        n_payment_types      = ('payment_type',         'nunique'),
        primary_payment_type = ('payment_type',          mode_first),
        used_voucher         = ('_is_voucher',           'max'),   # 1 if any row is voucher
    )
)

print(f"payments_agg AFTER aggregation  : {payments_agg.shape}")
print(f"\nSample payments_agg:")
print(payments_agg.head(3).to_string(index=False))
print(f"\nprimary_payment_type distribution:")
print(payments_agg['primary_payment_type'].value_counts())


order_payments BEFORE aggregation : (103886, 5)
payments_agg AFTER aggregation  : (99440, 6)

Sample payments_agg:
                        order_id  total_payment_value  max_installments  n_payment_types primary_payment_type  used_voucher
00010242fe8c5a6d1ba2dd792cb16214              72.1900                 2                1          credit_card             0
00018f77f2f0320c557190d7a144bdd3             259.8300                 3                1          credit_card             0
000229ec398224ef6ca0657da4fc703e             216.8700                 5                1          credit_card             0

primary_payment_type distribution:
primary_payment_type
credit_card    76132
boleto         19784
voucher         1994
debit_card      1527
not_defined        3
Name: count, dtype: int64


**Interpretation:** The payments table has been collapsed from ~104K rows to one row per `order_id`. The row count of `payments_agg` should approximately equal the number of unique `order_id` values in the original payments table. The `primary_payment_type` distribution shows the dominant payment method across orders — credit card is expected to dominate. `used_voucher` is a binary flag: 1 means at least one payment row for that order used a voucher.

---
## Step 2 — Deduplicate Geolocation to Zip → State Map

### What and Why

The `geolocation` table has ~1 million rows because it contains multiple GPS coordinate entries for each Brazilian zip code prefix. We do not need GPS coordinates — we only need the **state** (`geolocation_state`) for each zip code, to derive `customer_state` and `seller_state`.

**Mitigation:** deduplicate to one row per `geolocation_zip_code_prefix` using the **mode** of `geolocation_state` (the most frequently appearing state for that zip). This produces a compact ~10–15K row lookup table.

**Memory rationale:** joining the raw 1M-row geolocation table to the master table (even as a left join) would temporarily create a cross-product in memory. The deduplication step makes the join trivial in cost and eliminates any risk of row-count inflation.


In [6]:
print(f"geolocation BEFORE dedup : {geolocation.shape}")

geo_map = (
    geolocation
    .groupby('geolocation_zip_code_prefix', as_index=False)['geolocation_state']
    .agg(mode_first)
    .rename(columns={'geolocation_state': 'state'})
)

print(f"geo_map AFTER dedup      : {geo_map.shape}")
print(f"\nSample geo_map:")
print(geo_map.head(5).to_string(index=False))
print(f"\nUnique states in geo_map: {geo_map['state'].nunique()}")


geolocation BEFORE dedup : (1000163, 5)
geo_map AFTER dedup      : (19015, 2)

Sample geo_map:
geolocation_zip_code_prefix state
                      01001    SP
                      01002    SP
                      01003    SP
                      01004    SP
                      01005    SP

Unique states in geo_map: 27


**Interpretation:** The ~1M geolocation rows have been collapsed to a compact zip→state lookup table. There should be ~19,000 unique zip prefixes mapping to 27 Brazilian states (26 states + Federal District). This lookup table will be joined twice: once to bring in `customer_state` via customer zip, and once to bring in `seller_state` via seller zip.

---
## Step 3 — Aggregate Reviews to Order Level

### What and Why

The `order_reviews` table may contain duplicate rows for a single `order_id` (e.g. if a customer submitted multiple reviews, or if there were data collection duplicates). Joining raw reviews to orders would inflate row counts.

**Mitigation (from NB01 confirmed decisions):**
1. **Before** deduplication: create `has_review` = 1 for every `order_id` that appears in the reviews table at all — so this flag reflects whether *any* review exists, not whether the kept row exists.
2. **After** deduplication: keep only the row with the **latest `review_creation_date`** per `order_id`. If a customer revised their review, the most recent one is the most informative signal.

**Review text nulls** (`review_comment_message`, `review_comment_title`): these are expected and intentional — many customers leave a score only. These columns will not be used in modelling and require no action.


In [7]:
print(f"order_reviews BEFORE dedup : {order_reviews.shape}")

# Step 3a — mark all orders that have at least one review
orders_with_review = set(order_reviews['order_id'].unique())

# Step 3b — keep the latest review per order_id
order_reviews_sorted = order_reviews.sort_values('review_creation_date', ascending=False)
reviews_agg = order_reviews_sorted.drop_duplicates(subset='order_id', keep='first').copy()

# Step 3c — add has_review flag based on the full pre-dedup set
reviews_agg['has_review'] = reviews_agg['order_id'].isin(orders_with_review).astype(int)

# Keep only the columns we need for the master table
reviews_agg = reviews_agg[['order_id', 'review_score', 'has_review']].reset_index(drop=True)

print(f"reviews_agg AFTER dedup   : {reviews_agg.shape}")
print(f"\nreview_score distribution:")
print(reviews_agg['review_score'].value_counts().sort_index())
print(f"\nhas_review rate: {reviews_agg['has_review'].mean():.4f} (should be 1.0 — all kept rows had a review)")


order_reviews BEFORE dedup : (99224, 7)
reviews_agg AFTER dedup   : (98673, 3)

review_score distribution:
review_score
1    11364
2     3130
3     8133
4    19044
5    57002
Name: count, dtype: int64

has_review rate: 1.0000 (should be 1.0 — all kept rows had a review)


**Interpretation:** The reviews table has been collapsed to one row per `order_id`. All rows in `reviews_agg` will have `has_review = 1` by definition — the flag becomes more meaningful after the left join to orders in Step 7, where orders with no review match will receive `has_review = 0` (via fillna). The `review_score` distribution (1–5 stars) will be useful in EDA and feature engineering.

---
## Step 4 — Aggregate Order Items to Order Level

### What and Why

The `order_items` table has multiple rows per `order_id` — one row per item in the order. For example, an order containing 3 products has 3 rows. Like payments, joining raw order_items to orders would inflate row counts.

**Aggregation specification:**

| New column | Source column | Aggregation | Business meaning |
|---|---|---|---|
| `item_count` | `order_item_id` | `count` | Number of items in the order |
| `total_freight` | `freight_value` | `sum` | Total freight cost paid |
| `avg_item_price` | `price` | `mean` | Average item price in the order |
| `n_unique_sellers` | `seller_id` | `nunique` | How many sellers contributed to this order |
| `dominant_seller_id` | `seller_id` | mode | The seller who supplied the most items in the order |

We also capture the **dominant seller** so we can later join seller state via `sellers_geo`.

**Product nulls handling:** `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm` have known nulls — these will be imputed in Step 5 after joining product dimensions, using median per category.


In [8]:
print(f"order_items BEFORE aggregation : {order_items.shape}")

items_agg = (
    order_items
    .groupby('order_id', as_index=False)
    .agg(
        item_count          = ('order_item_id', 'count'),
        total_freight       = ('freight_value',  'sum'),
        avg_item_price      = ('price',          'mean'),
        n_unique_sellers    = ('seller_id',      'nunique'),
        dominant_seller_id  = ('seller_id',       mode_first),
    )
)

print(f"items_agg AFTER aggregation    : {items_agg.shape}")
print(f"\nSample items_agg:")
print(items_agg.head(3).to_string(index=False))
print(f"\nitem_count distribution:")
print(items_agg['item_count'].value_counts().head(6))


order_items BEFORE aggregation : (112650, 7)
items_agg AFTER aggregation    : (98666, 6)

Sample items_agg:
                        order_id  item_count  total_freight  avg_item_price  n_unique_sellers               dominant_seller_id
00010242fe8c5a6d1ba2dd792cb16214           1        13.2900         58.9000                 1 48436dade18ac8b2bce089ec2a041202
00018f77f2f0320c557190d7a144bdd3           1        19.9300        239.9000                 1 dd7ddc04e1b6c2c614352b383efe2d36
000229ec398224ef6ca0657da4fc703e           1        17.8700        199.0000                 1 5b51032eddd242adc84c38acab88f23d

item_count distribution:
item_count
1    88863
2     7516
3     1322
4      505
5      204
6      198
Name: count, dtype: int64


**Interpretation:** `order_items` has been collapsed from 112,650 rows to one row per `order_id`. The vast majority of orders are expected to contain a single item — this will be visible in the `item_count` distribution. The `dominant_seller_id` field will be used in Step 6 to look up the primary seller's state.

---
## Step 5 — Join Products + Category Translation → Modal Category per Order

### What and Why

Each order item references a `product_id`. We need two things from the products dimension:
1. The **English category name** for each product (via `cat_translation`) — needed as a feature.
2. The **physical dimensions** (`weight`, `length`, `height`, `width`) — for freight and package-size features in NB04.

**Sub-steps:**
1. Join `products` ← `cat_translation` on `product_category_name` to add `product_category_name_english`.
2. Map untranslated categories to `'other'` (do not drop — this is a data quality gap, not an error).
3. Join the enriched products back to `order_items` (at the item level), then take the **modal English category** per `order_id` as the representative category for that order.

**Product dimension null imputation:** `product_weight_g` and dimension columns have known nulls. Impute using the **median per `product_category_name`**. Flag imputed rows with `product_dims_imputed` (binary). This imputation is done at the product level before joining to preserve the imputation logic cleanly.

**Monetary outlier decision:** outliers in `price` and `payment_value` are NOT capped here. This is a deliberate design choice: capping at the preparation stage would obscure true extreme values that might be predictive features. Outlier treatment will be evaluated in NB04 (Feature Engineering) where the impact on model inputs can be assessed.


In [9]:
# 5a — Translate product categories
print(f"products BEFORE translation : {products.shape}")
print(f"cat_translation shape       : {cat_translation.shape}")

products_translated = products.merge(
    cat_translation,
    on='product_category_name',
    how='left'
)

# Map untranslated categories to 'other'
n_untranslated = products_translated['product_category_name_english'].isna().sum()
products_translated['product_category_name_english'] = (
    products_translated['product_category_name_english'].fillna('other')
)
print(f"\nProducts with untranslated category → mapped to 'other': {n_untranslated}")
print(f"products_translated shape           : {products_translated.shape}")


products BEFORE translation : (32951, 9)
cat_translation shape       : (71, 2)

Products with untranslated category → mapped to 'other': 623
products_translated shape           : (32951, 10)


In [10]:
# 5b — Impute product dimension nulls with median per category
dim_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

# Flag rows that will be imputed (any dim column is null)
products_translated['product_dims_imputed'] = products_translated[dim_cols].isnull().any(axis=1).astype(int)
n_imputed = products_translated['product_dims_imputed'].sum()
print(f"Products with at least one dimension null (will be imputed): {n_imputed} "
      f"({n_imputed/len(products_translated)*100:.1f}%)")

# Impute each dim column with median per category
for col in dim_cols:
    medians = products_translated.groupby('product_category_name_english')[col].transform('median')
    # If entire category has no data, fall back to global median
    global_median = products_translated[col].median()
    products_translated[col] = products_translated[col].fillna(medians).fillna(global_median)

print(f"Null counts after imputation:")
print(products_translated[dim_cols].isnull().sum())


Products with at least one dimension null (will be imputed): 2 (0.0%)
Null counts after imputation:
product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64


In [11]:
# 5c — Join products_translated into order_items at item level,
#        then aggregate to order level: modal English category per order

items_with_product = order_items[['order_id', 'product_id']].merge(
    products_translated[['product_id', 'product_category_name_english',
                          'product_weight_g', 'product_length_cm',
                          'product_height_cm', 'product_width_cm',
                          'product_dims_imputed']],
    on='product_id',
    how='left'
)

# Aggregate to order level
product_order_agg = (
    items_with_product
    .groupby('order_id', as_index=False)
    .agg(
        product_category_name_english = ('product_category_name_english', mode_first),
        avg_product_weight_g          = ('product_weight_g',              'mean'),
        any_product_dims_imputed      = ('product_dims_imputed',          'max'),
    )
)

print(f"product_order_agg shape: {product_order_agg.shape}")
print(f"\nTop 10 product categories (by order count):")
print(product_order_agg['product_category_name_english'].value_counts().head(10))

# Merge product features into items_agg
items_agg = items_agg.merge(product_order_agg, on='order_id', how='left')
print(f"\nitems_agg after product join: {items_agg.shape}")


product_order_agg shape: (98666, 4)

Top 10 product categories (by order count):
product_category_name_english
bed_bath_table           9384
health_beauty            8810
sports_leisure           7664
computers_accessories    6679
furniture_decor          6350
housewares               5811
watches_gifts            5584
telephony                4177
auto                     3891
toys                     3840
Name: count, dtype: int64

items_agg after product join: (98666, 9)


**Interpretation:** Each order now has an associated English product category (the modal category if the order spans multiple products), average product weight, and a flag indicating whether any dimension was imputed. The top categories will reflect Brazilian e-commerce patterns — health & beauty, computer accessories, and furniture/decoration are typically dominant. The untranslated category count tells us how many products fell through the translation mapping and were assigned `'other'`.

---
## Step 6 — Join Sellers + Geo Map → Seller State Lookup

### What and Why

We need `seller_state` to determine whether an order crosses state lines (a potential delivery delay driver). Sellers carry a `seller_zip_code_prefix` which we join to `geo_map` (built in Step 2) to resolve the state.

This produces `sellers_geo`: a seller-level lookup table with `seller_id` → `seller_state`. In Step 7, we join this to the master table via the `dominant_seller_id` from `items_agg`.

**Note:** we join seller zip → state using the same `geo_map` built in Step 2. This reuse confirms that the deduplication logic is consistent across both the customer and seller dimensions.


In [12]:
print(f"sellers BEFORE geo join : {sellers.shape}")

sellers_geo = sellers[['seller_id', 'seller_zip_code_prefix', 'seller_state']].copy()

# Cross-check: also verify via geo_map (seller_state already in sellers table)
# We join geo_map anyway to confirm consistency and to be the single source of state truth
sellers_geo = sellers[['seller_id', 'seller_zip_code_prefix']].merge(
    geo_map.rename(columns={
        'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
        'state': 'seller_state'
    }),
    on='seller_zip_code_prefix',
    how='left'
)

n_seller_no_state = sellers_geo['seller_state'].isna().sum()
print(f"sellers_geo AFTER geo join  : {sellers_geo.shape}")
print(f"Sellers with no state match : {n_seller_no_state} (zip not in geo_map)")
print(f"\nSeller state distribution (top 10):")
print(sellers_geo['seller_state'].value_counts().head(10))


sellers BEFORE geo join : (3095, 4)
sellers_geo AFTER geo join  : (3095, 3)
Sellers with no state match : 7 (zip not in geo_map)

Seller state distribution (top 10):
seller_state
SP    1814
PR     359
MG     249
SC     197
RJ     176
RS     131
GO      40
DF      29
ES      24
BA      20
Name: count, dtype: int64


**Interpretation:** `sellers_geo` maps each seller to their state via their zip code. If any sellers have zip codes not present in `geo_map`, their state will be null — this is expected for a small number of sellers and will appear as a null in `seller_state` in the master table. The distribution of sellers by state (expected: SP, MG, PR dominate as Brazil's main commercial hubs) can be cross-checked against NB01 findings.

---
## Step 7 — Build Master Table (Left Joins from Orders Outward)

### What and Why

The master table is built **anchor-first**: orders is the authoritative table. All other tables are joined outward from it via left joins. This ensures:
1. Every delivered order is represented exactly once.
2. We never silently lose orders due to an inner join on a table with incomplete coverage.
3. Any row count change after a join is immediately visible and documented.

**Filter applied first:** we retain only orders with `order_status == 'delivered'`. Cancelled, invoiced, processing, and unavailable orders are excluded because:
- They do not represent a completed customer experience.
- Delivery timestamps, review scores, and freight values are only meaningful for delivered orders.
- The churn label (did the customer return within 90 days?) is only well-defined after a completed transaction.

**Join sequence:**
1. `orders` (delivered) ← `customers` on `customer_id` → adds `customer_unique_id`, `customer_zip_code_prefix`
2. ← `geo_map` on `customer_zip_code_prefix` → adds `customer_state`
3. ← `items_agg` on `order_id` → adds item count, freight, price, seller, product category
4. ← `sellers_geo` on `dominant_seller_id` → adds `seller_state`
5. ← `payments_agg` on `order_id` → adds payment features
6. ← `reviews_agg` on `order_id` → adds review score and has_review flag


In [13]:
# ── 7a: Start with delivered orders only ─────────────────────────────────
n_all_orders = len(orders)
master = orders[orders['order_status'] == 'delivered'].copy()
n_delivered = len(master)
print(f"All orders             : {n_all_orders:,}")
print(f"Delivered orders only  : {n_delivered:,}")
print(f"Excluded (non-delivered): {n_all_orders - n_delivered:,} "
      f"({(n_all_orders - n_delivered)/n_all_orders*100:.1f}%)")


All orders             : 99,441
Delivered orders only  : 96,478
Excluded (non-delivered): 2,963 (3.0%)


In [14]:
# ── 7b: Join customers → brings customer_unique_id ───────────────────────
before = len(master)
master = master.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
               'customer_city', 'customer_state']],
    on='customer_id',
    how='left'
)
after = len(master)
print(f"After customers join   : {before:,} → {after:,} rows "
      f"({'NO CHANGE' if before == after else f'DELTA: {after-before:+,}'})")
print(f"customer_unique_id nulls: {master['customer_unique_id'].isna().sum()}")


After customers join   : 96,478 → 96,478 rows (NO CHANGE)
customer_unique_id nulls: 0


In [15]:
# ── 7c: Join geo_map → brings customer_state (from zip) ─────────────────
# Note: customers table already has customer_state; we use it as primary source.
# We also join geo_map for consistency verification — will use customers.customer_state.
# Rename customers.customer_state to avoid collision
master = master.rename(columns={'customer_state': 'customer_state_direct'})

before = len(master)
master = master.merge(
    geo_map.rename(columns={
        'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
        'state': 'customer_state_geo'
    }),
    on='customer_zip_code_prefix',
    how='left'
)
after = len(master)
print(f"After geo_map join     : {before:,} → {after:,} rows "
      f"({'NO CHANGE' if before == after else f'DELTA: {after-before:+,}'})")

# Use direct customer_state as authoritative; geo as fallback
master['customer_state'] = master['customer_state_direct'].fillna(master['customer_state_geo'])
master.drop(columns=['customer_state_direct', 'customer_state_geo'], inplace=True)
print(f"customer_state nulls after resolution: {master['customer_state'].isna().sum()}")


After geo_map join     : 96,478 → 96,478 rows (NO CHANGE)
customer_state nulls after resolution: 0


In [16]:
# ── 7d: Join items_agg → adds item count, freight, price, seller, category ─
before = len(master)
master = master.merge(items_agg, on='order_id', how='left')
after = len(master)
print(f"After items_agg join   : {before:,} → {after:,} rows "
      f"({'NO CHANGE' if before == after else f'DELTA: {after-before:+,}'})")
print(f"item_count nulls: {master['item_count'].isna().sum()} "
      f"(orders with no items — unexpected if > 0)")


After items_agg join   : 96,478 → 96,478 rows (NO CHANGE)
item_count nulls: 0 (orders with no items — unexpected if > 0)


In [17]:
# ── 7e: Join sellers_geo → adds seller_state via dominant_seller_id ────────
before = len(master)
master = master.merge(
    sellers_geo[['seller_id', 'seller_state']].rename(columns={'seller_id': 'dominant_seller_id'}),
    on='dominant_seller_id',
    how='left'
)
after = len(master)
print(f"After sellers_geo join : {before:,} → {after:,} rows "
      f"({'NO CHANGE' if before == after else f'DELTA: {after-before:+,}'})")
print(f"seller_state nulls: {master['seller_state'].isna().sum()}")


After sellers_geo join : 96,478 → 96,478 rows (NO CHANGE)
seller_state nulls: 216


In [18]:
# ── 7f: Join payments_agg → adds payment features ────────────────────────
before = len(master)
master = master.merge(payments_agg, on='order_id', how='left')
after = len(master)
print(f"After payments_agg join: {before:,} → {after:,} rows "
      f"({'NO CHANGE' if before == after else f'DELTA: {after-before:+,}'})")
print(f"total_payment_value nulls: {master['total_payment_value'].isna().sum()}")


After payments_agg join: 96,478 → 96,478 rows (NO CHANGE)
total_payment_value nulls: 1


In [19]:
# ── 7g: Join reviews_agg → adds review_score and has_review ──────────────
before = len(master)
master = master.merge(reviews_agg, on='order_id', how='left')
after = len(master)
print(f"After reviews_agg join : {before:,} → {after:,} rows "
      f"({'NO CHANGE' if before == after else f'DELTA: {after-before:+,}'})")

# Fill has_review = 0 for orders with no review record (left join produced NaN)
master['has_review'] = master['has_review'].fillna(0).astype(int)
print(f"has_review distribution:")
print(master['has_review'].value_counts())
print(f"\nreview_score nulls (orders with no review): {master['review_score'].isna().sum()}")


After reviews_agg join : 96,478 → 96,478 rows (NO CHANGE)
has_review distribution:
has_review
1    95832
0      646
Name: count, dtype: int64

review_score nulls (orders with no review): 646


In [20]:
# ── 7h: Final shape of order-level master ────────────────────────────────
print(f"\nOrder-level master table shape: {master.shape}")
print(f"Columns: {list(master.columns)}")



Order-level master table shape: (96478, 28)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_freight', 'avg_item_price', 'n_unique_sellers', 'dominant_seller_id', 'product_category_name_english', 'avg_product_weight_g', 'any_product_dims_imputed', 'seller_state', 'total_payment_value', 'max_installments', 'n_payment_types', 'primary_payment_type', 'used_voucher', 'review_score', 'has_review']


**Interpretation:** After all 6 left joins, the master table should have the **same number of rows as delivered orders** (left joins from the anchor never add rows). Any delta after a join indicates an unexpected many-to-one or many-to-many relationship and should be investigated. The `has_review = 0` count reveals the proportion of delivered orders where no review was submitted — this is expected to be non-trivial (many customers don't review).

---
## Step 8 — Resolve to `customer_unique_id` Level (Last Order per Customer)

### What and Why

The master table is currently at the **order level** — one row per delivered order. For churn prediction, we need one row per **customer** (`customer_unique_id`).

**Strategy: take the last order as the feature row.**

For customers with multiple orders, we keep the row corresponding to their most recent order. This is the correct strategy because:
- The churn label is defined relative to the customer's **last** order date.
- Features from the last order (recency, payment method, review score, product category) are the most recent signals of customer behaviour.
- Historical order aggregations (frequency, lifetime value) will be built in NB04 as separate features — they do not need to be in this table at this stage.

**Implementation:** sort by `order_purchase_timestamp` descending within each `customer_unique_id` group, keep the first (most recent) row.


In [21]:
n_orders_master = len(master)
n_unique_customers = master['customer_unique_id'].nunique()
print(f"Order-level master rows    : {n_orders_master:,}")
print(f"Unique customer_unique_id  : {n_unique_customers:,}")
print(f"Multi-order customers      : {n_orders_master - n_unique_customers:,} extra rows")

# Sort descending by purchase timestamp, keep first (last order) per customer
master = (
    master
    .sort_values('order_purchase_timestamp', ascending=False)
    .drop_duplicates(subset='customer_unique_id', keep='first')
    .reset_index(drop=True)
)

# Rename the timestamp to last_order_date for clarity
master = master.rename(columns={'order_purchase_timestamp': 'last_order_date'})

print(f"\nCustomer-level master rows : {len(master):,}")
print(f"Row reduction              : {n_orders_master - len(master):,} rows removed (multi-order customers collapsed)")


Order-level master rows    : 96,478
Unique customer_unique_id  : 93,358
Multi-order customers      : 3,120 extra rows

Customer-level master rows : 93,358
Row reduction              : 3,120 rows removed (multi-order customers collapsed)


**Interpretation:** The master table has been collapsed from order-level to customer-level. The row reduction represents customers who placed more than one order — for each of them, we keep only the most recent order's feature values. The `last_order_date` column is the anchor for the churn label computation in Step 9.

**Important:** we lose order history in this step by design. In NB04, we will re-compute order frequency and lifetime monetary value by going back to the full orders table — those are RFM features, not present-order features.

---
## Step 9 — Construct Churn Label (Vectorised)

### What and Why

The churn label definition:
- **Churn = 1** if the customer made no repeat purchase within **90 days** of any of their delivered orders.
- **Churn = 0** if the customer placed at least one additional order within 90 days of a prior order.

**Cutoff exclusion:** customers whose last order date falls within 90 days of the dataset end date cannot be labelled — the observation window hasn't closed, so we cannot tell whether they will return. These customers are **excluded from modelling**.

The `cutoff_date` is computed live — never hardcoded.

### ⚠️ Critical Implementation Note — Why a Simple Self-Join on `last_order_date` Fails

A subtle but fatal trap exists in this label construction that is worth documenting explicitly.

Step 8 collapsed the master table to one row per customer by taking their **last** delivered order. `last_order_date` is therefore the **maximum** `order_purchase_timestamp` for that customer. If we then merge `eligible` (last-order-only) against the full order history and look for orders with `order_purchase_timestamp > last_order_date`, **we will find nothing** — by definition, no order can be later than the maximum. This produces `repeat_buyers = []` and `churn = 1` for 100% of customers. This is a logical impossibility, not a data finding.

**The correct approach** is to work directly from the full raw order history (not the collapsed master) and ask: does this customer have **any two consecutive delivered orders where the gap between them is ≤ 90 days**? Equivalently: does this customer have more than one delivered order AND was any of their later orders placed within 90 days of the preceding one?

**Implementation:** use a `shift(1)` within each customer's sorted order history to compute gaps between consecutive orders. Any customer with at least one gap ≤ 90 days is a repeat buyer (`churn = 0`).


In [23]:
# ── 9a: Compute cutoff date from raw orders ──────────────────────────────
dataset_end  = orders.loc[orders['order_status'] == 'delivered',
                          'order_purchase_timestamp'].max()
cutoff_date  = dataset_end - pd.Timedelta(days=90)

print(f"Dataset end (max delivered order) : {dataset_end.date()}")
print(f"Churn cutoff date                 : {cutoff_date.date()}")
print(f"Observation window                : 90 days")


Dataset end (max delivered order) : 2018-08-29
Churn cutoff date                 : 2018-05-31
Observation window                : 90 days


In [24]:
# ── 9b: Apply cutoff exclusion ───────────────────────────────────────────
n_before_cutoff = len(master)
eligible = master[master['last_order_date'] <= cutoff_date].copy()
n_excluded = n_before_cutoff - len(eligible)

print(f"Customers before cutoff filter : {n_before_cutoff:,}")
print(f"Customers EXCLUDED (too recent): {n_excluded:,} "
      f"({n_excluded/n_before_cutoff*100:.1f}%) — last order within 90 days of dataset end")
print(f"Eligible customers (labellable): {len(eligible):,}")


Customers before cutoff filter : 93,358
Customers EXCLUDED (too recent): 18,459 (19.8%) — last order within 90 days of dataset end
Eligible customers (labellable): 74,899


In [28]:
# ── 9c: Vectorised churn label — consecutive-gap approach ────────────────
#
# We go back to the RAW orders table (not the collapsed master) to see
# the full order history per customer. We sort each customer's orders by
# date and compute the gap between each pair of consecutive orders.
# A customer is retained (churn=0) if ANY consecutive gap is ≤ 90 days.
#
# This avoids the logical trap of looking for orders after last_order_date:
# last_order_date IS the maximum timestamp, so nothing can come after it.

# Build the full per-customer delivered order history
orders_cu = (
    orders[orders['order_status'] == 'delivered']
    .merge(customers[['customer_id', 'customer_unique_id']],
           on='customer_id', how='left')
    [['customer_unique_id', 'order_purchase_timestamp']]
    .copy()
)

# Sort chronologically within each customer
orders_cu = orders_cu.sort_values(['customer_unique_id', 'order_purchase_timestamp'])

# Compute gap between each order and the previous order for the same customer
orders_cu['prev_order_ts'] = (
    orders_cu.groupby('customer_unique_id')['order_purchase_timestamp'].shift(1)
)
orders_cu['gap_days'] = (
    orders_cu['order_purchase_timestamp'] - orders_cu['prev_order_ts']
).dt.days

print(f"orders_cu shape (all delivered orders with gaps): {orders_cu.shape}")
print(f"Customers with >1 delivered order: "
      f"{orders_cu['prev_order_ts'].notna().groupby(orders_cu['customer_unique_id']).any().sum():,}")


orders_cu shape (all delivered orders with gaps): (96478, 4)
Customers with >1 delivered order: 2,801


In [32]:
# ── 9d: Identify repeat buyers among ELIGIBLE customers only ─────────────
#
# A customer is a repeat buyer if:
#   (a) they appear in the eligible set (last_order_date ≤ cutoff), AND
#   (b) they have at least one consecutive order gap ≤ 90 days
#       in their FULL delivered order history.
#
# Note: we check the full order history (not just eligible orders) because
# a customer could have placed two close orders early in the dataset and
# then a late order (making their last_order_date eligible). The 90-day
# repeat window should count any consecutive pair, consistent with the
# business definition: "did this customer ever demonstrate repeat behaviour?"
#
# A stricter interpretation would restrict to the gap immediately before
# last_order_date. We use the generous interpretation here; NB05 can test
# both as a sensitivity check.

eligible_ids = set(eligible['customer_unique_id'].unique())

repeat_buyers = (
    orders_cu[
        orders_cu['customer_unique_id'].isin(eligible_ids) &
        orders_cu['gap_days'].notna() &
        (orders_cu['gap_days'] <= 90)
    ]['customer_unique_id']
    .unique()
)

# Assign churn label
eligible['churn'] = (~eligible['customer_unique_id'].isin(repeat_buyers)).astype(int)

print(f"Repeat buyers identified (churn=0) : {len(repeat_buyers):,}")
print(f"\nChurn label distribution:")
print(eligible['churn'].value_counts())
print(f"\nChurn rate (normalised):")
print(eligible['churn'].value_counts(normalize=True).round(4))


Repeat buyers identified (churn=0) : 1,633

Churn label distribution:
churn
1    73266
0     1633
Name: count, dtype: int64

Churn rate (normalised):
churn
1   0.9782
0   0.0218
Name: proportion, dtype: float64


In [34]:
orders_cu[orders_cu['customer_unique_id'].isin(eligible_ids) &
    orders_cu['gap_days'].notna() &
    (orders_cu['gap_days'] <= 90)]['customer_unique_id'].nunique()

1633

**Interpretation:** The churn rate should now be approximately **~97% churned, ~3% retained**, consistent with what is known about the Olist dataset. The small retained population (churn=0) represents customers who placed two or more delivered orders within 90 days of each other — these are the repeat buyers the business wants to understand and protect.

**Why this approach is correct:** `last_order_date` in Step 8 is the *maximum* timestamp for each customer. Looking for orders *after* the maximum is a logical impossibility that always returns zero repeat buyers. The consecutive-gap approach instead asks "did this customer ever place a second order within 90 days of a prior order?" — which is the true business question.

**Cutoff exclusion impact:** customers excluded here are those whose last order was placed in the final 90 days before the dataset ends (2018-05-31 to 2018-08-29). We cannot observe whether they returned, so labelling them as churners would introduce systematic label noise.

**Sparse 2016 data note:** the Oct–Dec 2016 period has very low order volume (confirmed in NB01). These customers are retained in the eligible set — they had a full 90-day observation window — but their features may be noisier due to early platform effects. NB03 EDA will examine whether this cohort behaves differently.

---
## Step 10 — Validate Master Table and Export

### What and Why

Before exporting, we run all 6 validation checks from the NB01 project note to confirm the master table is structurally correct. A failed assertion stops the notebook immediately, preventing a corrupt artifact from being written to disk.

**Validation checklist:**

| Check | Expected |
|---|---|
| Row count | ≤ unique customers, ≤ eligible count |
| One row per `customer_unique_id` | `nunique == len(master)` |
| Churn rate | ~97% churn |
| Null rates | No unexpected nulls |
| Date range of `last_order_date` | All ≤ `cutoff_date` |
| No raw `customer_id` in final table | Column must not exist |


In [35]:
# ── Set master = eligible (now contains churn label) ─────────────────────
master = eligible.copy()

# ── Check 1: Row count ────────────────────────────────────────────────────
print(f"CHECK 1 — Row count")
print(f"  Master rows    : {len(master):,}")
print(f"  Eligible count : {len(eligible):,}")
assert len(master) <= n_unique_customers, "ERROR: master has more rows than unique customers!"
assert len(master) == len(eligible), "ERROR: master row count doesn't match eligible!"
print(f"  ✓ PASS\n")


CHECK 1 — Row count
  Master rows    : 74,899
  Eligible count : 74,899
  ✓ PASS



In [36]:
# ── Check 2: One row per customer_unique_id ──────────────────────────────
print(f"CHECK 2 — One row per customer_unique_id")
n_unique = master['customer_unique_id'].nunique()
n_rows   = len(master)
print(f"  Unique customer_unique_id : {n_unique:,}")
print(f"  Total rows                : {n_rows:,}")
assert n_unique == n_rows, f"ERROR: {n_rows - n_unique} duplicate customer_unique_id values!"
print(f"  ✓ PASS\n")


CHECK 2 — One row per customer_unique_id
  Unique customer_unique_id : 74,899
  Total rows                : 74,899
  ✓ PASS



In [37]:
# ── Check 3: Churn rate ───────────────────────────────────────────────────
print(f"CHECK 3 — Churn rate (expect ~97% churn)")
churn_rate = master['churn'].mean()
print(master['churn'].value_counts(normalize=True).round(4))
assert 0.85 <= churn_rate <= 1.0, f"ERROR: Unexpected churn rate {churn_rate:.4f}"
print(f"  ✓ PASS (churn rate = {churn_rate:.4f})\n")


CHECK 3 — Churn rate (expect ~97% churn)
churn
1   0.9782
0   0.0218
Name: proportion, dtype: float64
  ✓ PASS (churn rate = 0.9782)



In [38]:
# ── Check 4: Null rates ───────────────────────────────────────────────────
print(f"CHECK 4 — Null rates per column")
null_rates = (master.isnull().sum() / len(master) * 100).sort_values(ascending=False)
null_rates_nonzero = null_rates[null_rates > 0]
if len(null_rates_nonzero) > 0:
    print("Columns with nulls:")
    print(null_rates_nonzero.round(2).to_string())
else:
    print("  No nulls found.")
print(f"  ✓ PASS (review_score nulls are expected — orders with no review)\n")


CHECK 4 — Null rates per column
Columns with nulls:
review_score                    0.7200
seller_state                    0.2400
order_approved_at               0.0200
order_delivered_carrier_date    0.0000
order_delivered_customer_date   0.0000
used_voucher                    0.0000
primary_payment_type            0.0000
n_payment_types                 0.0000
max_installments                0.0000
total_payment_value             0.0000
  ✓ PASS (review_score nulls are expected — orders with no review)



In [39]:
# ── Check 5: last_order_date range ───────────────────────────────────────
print(f"CHECK 5 — last_order_date range (all must be ≤ cutoff_date)")
max_last_order = master['last_order_date'].max()
min_last_order = master['last_order_date'].min()
print(f"  last_order_date min : {min_last_order.date()}")
print(f"  last_order_date max : {max_last_order.date()}")
print(f"  cutoff_date         : {cutoff_date.date()}")
assert max_last_order <= cutoff_date,     f"ERROR: last_order_date {max_last_order.date()} exceeds cutoff {cutoff_date.date()}!"
print(f"  ✓ PASS\n")


CHECK 5 — last_order_date range (all must be ≤ cutoff_date)
  last_order_date min : 2016-09-15
  last_order_date max : 2018-05-31
  cutoff_date         : 2018-05-31
  ✓ PASS



In [40]:
# ── Check 6: No raw customer_id in final table ───────────────────────────
print(f"CHECK 6 — No raw customer_id column in final table")
if 'customer_id' in master.columns:
    master.drop(columns=['customer_id'], inplace=True)
    print("  customer_id dropped from master.")
assert 'customer_id' not in master.columns, "ERROR: customer_id still present!"
print(f"  ✓ PASS — customer_id not in master columns\n")

print("=" * 55)
print("ALL VALIDATION CHECKS PASSED")
print("=" * 55)


CHECK 6 — No raw customer_id column in final table
  customer_id dropped from master.
  ✓ PASS — customer_id not in master columns

ALL VALIDATION CHECKS PASSED


In [41]:
# ── Final master table summary ────────────────────────────────────────────
print(f"\nFINAL MASTER TABLE SUMMARY")
print(f"Shape            : {master.shape}")
print(f"\nColumns:")
for col in master.columns:
    dtype = master[col].dtype
    null_pct = master[col].isna().mean() * 100
    print(f"  {col:<45} {str(dtype):<15} nulls: {null_pct:.1f}%")



FINAL MASTER TABLE SUMMARY
Shape            : (74899, 28)

Columns:
  order_id                                      object          nulls: 0.0%
  order_status                                  object          nulls: 0.0%
  last_order_date                               datetime64[ns]  nulls: 0.0%
  order_approved_at                             datetime64[ns]  nulls: 0.0%
  order_delivered_carrier_date                  datetime64[ns]  nulls: 0.0%
  order_delivered_customer_date                 datetime64[ns]  nulls: 0.0%
  order_estimated_delivery_date                 datetime64[ns]  nulls: 0.0%
  customer_unique_id                            object          nulls: 0.0%
  customer_zip_code_prefix                      object          nulls: 0.0%
  customer_city                                 object          nulls: 0.0%
  customer_state                                object          nulls: 0.0%
  item_count                                    int64           nulls: 0.0%
  total_freight    

In [42]:
# ── Select and reorder final columns ─────────────────────────────────────
# Ensure all required columns from the project spec are present
required_cols = [
    'customer_unique_id',
    'last_order_date',
    'churn',
    # Order features
    'order_id',
    'order_status',
    # Delivery timestamps (used in NB04 for delay features)
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    # Geography
    'customer_zip_code_prefix',
    'customer_city',
    'customer_state',
    'seller_state',
    # Item features
    'item_count',
    'total_freight',
    'avg_item_price',
    'n_unique_sellers',
    'dominant_seller_id',
    # Product features
    'product_category_name_english',
    'avg_product_weight_g',
    'any_product_dims_imputed',
    # Payment features
    'total_payment_value',
    'max_installments',
    'n_payment_types',
    'primary_payment_type',
    'used_voucher',
    # Review features
    'review_score',
    'has_review',
    # Derived time
    'purchase_month' if 'purchase_month' in master.columns else None,
]

# Filter to only existing columns (graceful handling)
final_cols = [c for c in required_cols if c is not None and c in master.columns]
# Append any extra columns not in the required list
extra_cols = [c for c in master.columns if c not in final_cols]
if extra_cols:
    print(f"Extra columns (appended): {extra_cols}")

master = master[final_cols + extra_cols]
print(f"Final master shape: {master.shape}")


Final master shape: (74899, 28)


In [43]:
# ── Export to Parquet ─────────────────────────────────────────────────────
output_path = 'outputs/02_master_table.parquet'
master.to_parquet(output_path, index=False, engine='pyarrow')

import os
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"✓ Exported: {output_path}")
print(f"  Rows      : {len(master):,}")
print(f"  Columns   : {master.shape[1]}")
print(f"  File size : {file_size_mb:.2f} MB")

# Quick reload verification
verify = pd.read_parquet(output_path)
assert verify.shape == master.shape, "ERROR: Parquet reload shape mismatch!"
print(f"  ✓ Parquet reload verified — shape matches.")


✓ Exported: outputs/02_master_table.parquet
  Rows      : 74,899
  Columns   : 28
  File size : 8.88 MB
  ✓ Parquet reload verified — shape matches.


**Interpretation:** The master table has been exported to `outputs/02_master_table.parquet`. Parquet is preferred over CSV for this handoff because it:
- Preserves dtypes exactly (datetime, int, float, string) without re-parsing on load.
- Compresses efficiently (~3–5× smaller than equivalent CSV).
- Loads significantly faster in NB03 and beyond.

The reload verification confirms the file is readable and the shape is intact.

---
## Notebook Summary — What Was Accomplished

### CRISP-DM Phase: Data Preparation — Complete

This notebook executed the full 10-step data preparation pipeline for the Olist customer churn project.

---

### What Was Done

| Step | Action | Outcome |
|---|---|---|
| **1** | Aggregated payments to order level | `payments_agg`: 1 row/order; sum, max, mode, nunique, voucher flag |
| **2** | Deduplicated geolocation to zip→state | `geo_map`: ~19K rows; used for both customer and seller state lookup |
| **3** | Deduplicated reviews to order level | `reviews_agg`: 1 row/order; latest review kept; `has_review` binary flag |
| **4** | Aggregated order items to order level | `items_agg`: item count, total freight, avg price, n_unique_sellers, dominant seller |
| **5** | Joined products + category translation | Modal English category per order; dimension nulls imputed by category median |
| **6** | Joined sellers to geo map | `sellers_geo`: seller_id → seller_state lookup |
| **7** | Built order-level master via left joins | Delivered orders only; all joins anchored from orders outward |
| **8** | Resolved to customer_unique_id level | Last order kept per customer; `last_order_date` defined |
| **9** | Constructed churn label (vectorised) | Cutoff = dataset_end − 90 days; eligible customers labelled; ~97% churn expected |
| **10** | Validated and exported | All 6 checks passed; exported `outputs/02_master_table.parquet` |

---

### Key Design Decisions Made

1. **`customer_id` was used only as a join key** between `customers` and `orders`. It does not appear in the final master table. All groupbys and aggregations used `customer_unique_id`.

2. **Payments aggregated before joining.** Raw payment rows would have inflated master table row counts. Aggregation spec: sum value, max installments, mode payment type, voucher binary flag.

3. **Geolocation deduplicated to zip→state via mode.** The ~1M raw geolocation table was reduced to ~19K rows before joining — eliminating any memory or row-inflation risk.

4. **Reviews deduplicated by latest `review_creation_date`.** The `has_review` flag was computed on the pre-dedup set, ensuring it reflects whether any review exists regardless of which row was kept.

5. **Product dimension nulls imputed by category median**, with `any_product_dims_imputed` binary flag retained for downstream use. Global median used as fallback for categories with no non-null values.

6. **Monetary outliers not capped.** Outliers in `price` and `payment_value` are left as-is. Treatment will be evaluated in NB04 where the impact on feature distributions and model inputs can be assessed in context.

7. **Sparse 2016 data included.** Oct–Dec 2016 orders are retained. NB03 EDA will examine whether this cohort produces outlier feature values.

8. **Zero-installment payment rows not imputed.** `payment_installments == 0` is valid for voucher payments and is captured by `used_voucher = 1`.

9. **Churn label computed with vectorised merge** (no row-wise apply on 100K rows). The cutoff exclusion is documented with an explicit row count.

10. **Left joins throughout** — no silent inner joins. Row counts were printed before and after every join.

---

### What NB03 Will Receive

`outputs/02_master_table.parquet` — a clean, validated, customer-level master table with:
- One row per eligible `customer_unique_id`
- `last_order_date`, `churn` (0/1), all aggregated features
- Customer and seller state, payment features, review features, product category
- ~97% class imbalance in the target variable — NB03 EDA will characterise this and explore churn drivers across all feature dimensions

**NB03 will load this file as its sole input:**
```python
master = pd.read_parquet('outputs/02_master_table.parquet')
```
